In [ ]:
###############################################################################
# CELL 1: CONFIGURATION
###############################################################################
# Update ONLY this cell for different deployments.
# All other cells run without modification.
###############################################################################

import requests
import json
import time
import pandas as pd
from datetime import datetime

# --- Authentication (Service Principal) -------------------------------------
TENANT_ID     = "zzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzz"
CLIENT_ID     = "zzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzz"
CLIENT_SECRET = "zzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzz"
RESOURCE      = "https://analysis.windows.net/powerbi/api"
AUTHORITY     = f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token"

# --- Workspace Configuration ------------------------------------------------
DEV_WORKSPACE_ID = "547dab98-cdb5-405b-b73c-21f5d92486d8"
UAT_WORKSPACE_ID = "b34b7f5f-95e8-4e4e-a3e8-444334e0a6ab"

# --- Pipeline Configuration -------------------------------------------------
PIPELINE_NAME      = "AnalyticsBI_DevToUAT"
SOURCE_STAGE_ORDER = 0    # Development = 0, Test = 1, Production = 2

# --- Items to Deploy ---------------------------------------------------------
# Both semantic model and report deploy every time.
# Post-deployment auto-configures the UAT connection via service principal.
# To add more items, append to the relevant list below.
# -----------------------------------------------------------------------------
ITEMS_TO_DEPLOY = {
    "datasets": [
        {"sourceId": "a0461642-df45-442d-8383-671e60c356a4"}   # 06.Lakehouse View to PBI Version 04
    ],
    "reports": [
        {"sourceId": "c0403d79-2866-4edd-b60a-943986271421"}   # 06.Lakehouse View to PBI Version 04
    ]
}

# --- UAT Data Source Configuration -------------------------------------------
# Gateway and datasource IDs for the UAT cloud connection (LH_Marketing_CRM).
# Used by Phase 6 (Auto-Configuration) as a fallback if discovery fails.
# These were discovered via the Datasets - Get Datasources API.
# -----------------------------------------------------------------------------
UAT_GATEWAY_ID    = "61dc8e29-94e2-4cd6-a789-65bced2c0da8"
UAT_DATASOURCE_ID = "d77600ef-b62f-4b7a-b3a9-897f3aed09e5"

# --- Deployment Options ------------------------------------------------------
ALLOW_CREATE    = True     # Create item if it does not exist in UAT
ALLOW_OVERWRITE = True     # Overwrite item if it already exists in UAT

# --- Base URL ----------------------------------------------------------------
BASE_URL = "https://api.powerbi.com/v1.0/myorg"

# --- Summary -----------------------------------------------------------------
print("DEPLOYMENT NOTEBOOK: Dev to UAT")
print("=" * 70)
print(f"  Execution Time   : {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")
print(f"  Pipeline         : {PIPELINE_NAME}")
print(f"  Source (Dev)      : {DEV_WORKSPACE_ID}")
print(f"  Target (UAT)      : {UAT_WORKSPACE_ID}")
print(f"  Datasets          : {len(ITEMS_TO_DEPLOY.get('datasets', []))}")
print(f"  Reports           : {len(ITEMS_TO_DEPLOY.get('reports', []))}")
print(f"  Allow Create      : {ALLOW_CREATE}")
print(f"  Allow Overwrite   : {ALLOW_OVERWRITE}")
print("=" * 70)

StatementMeta(, 3d77b0e9-9802-4788-a5d5-9aeb4fc047cb, 41, Finished, Available, Finished, False)

DEPLOYMENT NOTEBOOK: Dev to UAT
  Execution Time   : 22/02/2026 15:54:04
  Pipeline         : AnalyticsBI_DevToUAT
  Source (Dev)      : 547dab98-cdb5-405b-b73c-21f5d92486d8
  Target (UAT)      : b34b7f5f-95e8-4e4e-a3e8-444334e0a6ab
  Datasets          : 1
  Reports           : 1
  Allow Create      : True
  Allow Overwrite   : True


In [ ]:
###############################################################################
# CELL 2: PHASE 1 — AUTHENTICATION
###############################################################################

print("PHASE 1: Authentication")
print("-" * 70)

body = {
    'client_id': CLIENT_ID,
    'scope': f'{RESOURCE}/.default',
    'client_secret': CLIENT_SECRET,
    'grant_type': 'client_credentials'
}

response = requests.post(AUTHORITY, data=body, headers={'Content-Type': 'application/x-www-form-urlencoded'})

if response.status_code != 200:
    print(f"  FAILED: {response.status_code}")
    print(f"  Error: {response.text}")
    raise Exception("Authentication failed -- check client secret expiry in Entra ID")

access_token = response.json()['access_token']
api_headers = {
    'Authorization': f'Bearer {access_token}',
    'Content-Type': 'application/json'
}

print("  Token acquired successfully")

StatementMeta(, 3d77b0e9-9802-4788-a5d5-9aeb4fc047cb, 42, Finished, Available, Finished, False)

PHASE 1: Authentication
----------------------------------------------------------------------
  Token acquired successfully


In [ ]:
###############################################################################
# CELL 3: PHASE 2 — PIPELINE SETUP
###############################################################################

print("\nPHASE 2: Pipeline Setup")
print("-" * 70)

# --- Find existing pipeline -------------------------------------------------
response = requests.get(f"{BASE_URL}/pipelines", headers=api_headers)
response.raise_for_status()

pipeline_id = None
for p in response.json().get('value', []):
    if p['displayName'] == PIPELINE_NAME:
        pipeline_id = p['id']
        print(f"  Found existing pipeline: {PIPELINE_NAME}")
        print(f"  Pipeline ID: {pipeline_id}")
        break

# --- Create pipeline if it does not exist -----------------------------------
if not pipeline_id:
    print(f"  Creating new pipeline: {PIPELINE_NAME}...")
    response = requests.post(
        f"{BASE_URL}/pipelines",
        headers=api_headers,
        json={"displayName": PIPELINE_NAME}
    )
    response.raise_for_status()
    pipeline_id = response.json()['id']
    print(f"  Pipeline created: {pipeline_id}")

# --- Check and assign workspaces to stages ----------------------------------
stages_response = requests.get(f"{BASE_URL}/pipelines/{pipeline_id}/stages", headers=api_headers)
stages_response.raise_for_status()
stages = stages_response.json().get('value', [])

workspace_map = {0: DEV_WORKSPACE_ID, 1: UAT_WORKSPACE_ID}
stage_names   = {0: "Development", 1: "Test/UAT", 2: "Production"}

for stage in stages:
    order = stage['order']
    ws_id = stage.get('workspaceId')

    if ws_id:
        print(f"  Stage {order} ({stage_names.get(order, 'Unknown')}): {ws_id}")
    elif order in workspace_map:
        print(f"  Assigning workspace to Stage {order}...")
        r = requests.post(
            f"{BASE_URL}/pipelines/{pipeline_id}/stages/{order}/assignWorkspace",
            headers=api_headers,
            json={"workspaceId": workspace_map[order]}
        )
        if r.status_code == 200:
            print(f"  Stage {order} ({stage_names.get(order, 'Unknown')}): Assigned")
        else:
            raise Exception(f"Could not assign workspace to Stage {order}: {r.status_code} -- {r.text[:200]}")
    else:
        print(f"  Stage {order} ({stage_names.get(order, 'Unknown')}): Not assigned")

print(f"\n  Pipeline ready: {pipeline_id}")

StatementMeta(, 3d77b0e9-9802-4788-a5d5-9aeb4fc047cb, 43, Finished, Available, Finished, False)


PHASE 2: Pipeline Setup
----------------------------------------------------------------------
  Found existing pipeline: AnalyticsBI_DevToUAT
  Pipeline ID: 897b66f2-ddbf-4c77-8c64-a2e7555ce1a4
  Stage 0 (Development): 547dab98-cdb5-405b-b73c-21f5d92486d8
  Stage 1 (Test/UAT): b34b7f5f-95e8-4e4e-a3e8-444334e0a6ab
  Stage 2 (Production): Not assigned

  Pipeline ready: 897b66f2-ddbf-4c77-8c64-a2e7555ce1a4


In [ ]:
###############################################################################
# CELL 4: PHASE 3 — PRE-DEPLOYMENT CHECK
###############################################################################

print("\nPHASE 3: Pre-Deployment Check")
print("-" * 70)

# --- List all items in Dev stage --------------------------------------------
artifacts_url = f"{BASE_URL}/pipelines/{pipeline_id}/stages/0/artifacts"
response = requests.get(artifacts_url, headers=api_headers)
response.raise_for_status()
dev_data = response.json()

for item_type in ['datasets', 'reports', 'dashboards', 'dataflows']:
    items = dev_data.get(item_type, [])
    if items:
        print(f"\n  {item_type.upper()} in Dev stage:")
        for item in items:
            artifact_id   = item.get('artifactId', 'N/A')
            artifact_name = item.get('artifactDisplayName', 'N/A')
            deploy_ids    = [d['sourceId'] for d in ITEMS_TO_DEPLOY.get(item_type, [])]
            marker        = " <-- DEPLOYING" if artifact_id in deploy_ids else ""
            print(f"    {artifact_name} | ID: {artifact_id}{marker}")

# --- Validate deployment manifest -------------------------------------------
print(f"\n  Deployment manifest validation:")
all_stage_ids = []
for item_type in ['datasets', 'reports']:
    all_stage_ids.extend([i['artifactId'] for i in dev_data.get(item_type, [])])

all_valid = True
for item_type, items in ITEMS_TO_DEPLOY.items():
    for item in items:
        if item['sourceId'] in all_stage_ids:
            print(f"VALID   {item_type}: {item['sourceId']}")
        else:
            print(f"MISSING {item_type}: {item['sourceId']} -- NOT FOUND in Dev stage")
            all_valid = False

if not all_valid:
    raise Exception("Some items not found in Dev stage -- check IDs in Cell 1")
else:
    print(f"\n  All items validated -- ready to deploy")

StatementMeta(, 3d77b0e9-9802-4788-a5d5-9aeb4fc047cb, 44, Finished, Available, Finished, False)


PHASE 3: Pre-Deployment Check
----------------------------------------------------------------------

  DATASETS in Dev stage:
    06.Lakehouse View to PBI Version 04 | ID: a0461642-df45-442d-8383-671e60c356a4 <-- DEPLOYING

  REPORTS in Dev stage:
    06.Lakehouse View to PBI Version 04 | ID: c0403d79-2866-4edd-b60a-943986271421 <-- DEPLOYING

  Deployment manifest validation:
VALID   datasets: a0461642-df45-442d-8383-671e60c356a4
VALID   reports: c0403d79-2866-4edd-b60a-943986271421

  All items validated -- ready to deploy


In [ ]:
# ============================================================================
# CELL 5: PHASE 4 — DEPLOY (Dev to UAT)
# ============================================================================

print("\nPHASE 4: Deploying Dev to UAT")
print("-" * 70)

# --- Auto-detect: check if UAT already has items ---------------------------
uat_artifacts_url = f"{BASE_URL}/pipelines/{pipeline_id}/stages/1/artifacts"
uat_check = requests.get(uat_artifacts_url, headers=api_headers)
uat_check.raise_for_status()
uat_data = uat_check.json()

uat_has_datasets = len(uat_data.get('datasets', [])) > 0
uat_has_reports  = len(uat_data.get('reports', [])) > 0

# --- Decide deployment mode ------------------------------------------------
if uat_has_datasets and uat_has_reports:
    deploy_mode = "REPORT ONLY"
    deploy_items = {
        "reports": ITEMS_TO_DEPLOY.get("reports", [])
    }
    is_first_deployment = False
    print(f"  Mode: {deploy_mode} (UAT has semantic model -- preserving connection)")
else:
    deploy_mode = "FULL (Dataset + Report)"
    deploy_items = {
        "datasets": ITEMS_TO_DEPLOY.get("datasets", []),
        "reports": ITEMS_TO_DEPLOY.get("reports", [])
    }
    is_first_deployment = True
    print(f"  Mode: {deploy_mode} (UAT is empty -- deploying everything)")

deploy_body = {
    "sourceStageOrder": SOURCE_STAGE_ORDER,
    "options": {
        "allowCreateArtifact": ALLOW_CREATE,
        "allowOverwriteArtifact": ALLOW_OVERWRITE,
        "allowTakeover": True
    },
    "note": f"Automated deployment ({deploy_mode}) -- {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}"
}

if "datasets" in deploy_items:
    deploy_body["datasets"] = deploy_items["datasets"]
if "reports" in deploy_items:
    deploy_body["reports"] = deploy_items["reports"]

print(f"  Source Stage   : {SOURCE_STAGE_ORDER} (Development)")
print(f"  Target Stage   : {SOURCE_STAGE_ORDER + 1} (Test/UAT)")
print(f"  Datasets       : {len(deploy_items.get('datasets', []))}")
print(f"  Reports        : {len(deploy_items.get('reports', []))}")
print(f"  Allow Create   : {ALLOW_CREATE}")
print(f"  Allow Overwrite: {ALLOW_OVERWRITE}")

response = requests.post(
    f"{BASE_URL}/pipelines/{pipeline_id}/deploy",
    headers=api_headers,
    json=deploy_body
)

if response.status_code == 202:
    operation = response.json()
    operation_id = operation['id']
    print(f"\n  Deployment accepted")
    print(f"  Operation ID: {operation_id}")
else:
    print(f"\n  Deployment failed: {response.status_code}")
    print(f"  Error: {response.text[:500]}")
    raise Exception("Deployment request failed")

StatementMeta(, 3d77b0e9-9802-4788-a5d5-9aeb4fc047cb, 45, Finished, Available, Finished, False)


PHASE 4: Deploying Dev to UAT
----------------------------------------------------------------------
  Mode: REPORT ONLY (UAT has semantic model -- preserving connection)
  Source Stage   : 0 (Development)
  Target Stage   : 1 (Test/UAT)
  Datasets       : 0
  Reports        : 1
  Allow Create   : True
  Allow Overwrite: True



  Deployment accepted
  Operation ID: e9d807ee-fae1-4b0c-88b9-b7fbddf5af64


In [ ]:
###############################################################################
# CELL 6: PHASE 5 — POLL FOR COMPLETION
###############################################################################

print("\nPHASE 5: Waiting for deployment to complete...")
print("-" * 70)

operation_url = f"{BASE_URL}/pipelines/{pipeline_id}/operations/{operation_id}"
max_wait      = 300
poll_interval = 5
elapsed       = 0

while elapsed < max_wait:
    response = requests.get(operation_url, headers=api_headers)
    response.raise_for_status()

    operation = response.json()
    status    = operation.get('status', 'Unknown')

    if status == 'Succeeded':
        print(f"\n  DEPLOYMENT SUCCEEDED")
        print(f"  Started : {operation.get('executionStartTime', 'N/A')}")
        print(f"  Ended   : {operation.get('executionEndTime', 'N/A')}")

        steps = operation.get('executionPlan', {}).get('steps', [])
        if steps:
            print(f"\n  Deployment Steps:")
            for step in steps:
                src = step.get('sourceAndTarget', {})
                print(f"    {src.get('sourceDisplayName', 'N/A')} "
                      f"({step.get('type', 'N/A')}) -- Status: {step.get('status', 'N/A')}")
        break

    elif status == 'Failed':
        print(f"\n  DEPLOYMENT FAILED")
        steps = operation.get('executionPlan', {}).get('steps', [])
        for step in steps:
            error = step.get('error', {})
            if error:
                print(f"  Error: {error.get('errorCode', 'N/A')}")
                print(f"  Details: {error.get('errorDetails', 'N/A')}")
        raise Exception("Deployment failed -- see error details above")

    else:
        print(f"  Polling... Status: {status} ({elapsed}s elapsed)")
        time.sleep(poll_interval)
        elapsed += poll_interval

if elapsed >= max_wait:
    print(f"\n  Timed out after {max_wait}s -- check pipeline in the Fabric portal")

StatementMeta(, 3d77b0e9-9802-4788-a5d5-9aeb4fc047cb, 46, Finished, Available, Finished, False)


PHASE 5: Waiting for deployment to complete...
----------------------------------------------------------------------

  DEPLOYMENT SUCCEEDED
  Started : 2026-02-22T15:54:34.773Z
  Ended   : 2026-02-22T15:54:35.9Z

  Deployment Steps:
    06.Lakehouse View to PBI Version 04 (ReportDeployment) -- Status: Succeeded


In [ ]:
###############################################################################
# CELL 7: PHASE 6 — POST-DEPLOYMENT AUTO-CONFIGURATION
###############################################################################
# After deployment, the UAT semantic model needs its data connection
# configured. This cell automates the full process:
#   1. Discover the UAT dataset ID from the pipeline artifacts
#   2. Service principal takes over the semantic model
#   3. Discover datasource and gateway IDs (dynamic, not hardcoded)
#   4. Bind the semantic model to the cloud connection
#   5. Wait for bind to propagate
#   6. Trigger a data refresh
#
# This eliminates all manual portal steps (Take Over, Gateway Settings,
# Maps To dropdown, Apply, Refresh).
###############################################################################

StatementMeta(, 3d77b0e9-9802-4788-a5d5-9aeb4fc047cb, 47, Finished, Available, Finished, False)

In [ ]:
# ============================================================================
# CELL 7: PHASE 6 — POST-DEPLOYMENT AUTO-CONFIGURATION
# ============================================================================
# Only runs on first-time deployments (when UAT was empty).
# On subsequent deployments (report-only), the connection is already
# configured and this cell is skipped.
#
# After first deployment, you must complete ONE manual step in the portal:
#   1. Go to UAT semantic model Settings
#   2. Gateway and cloud connections
#   3. Maps to: select LH_Marketing_CRM
#   4. Apply
#
# This manual step is required because the Power BI REST API cannot map
# a semantic model to a named shareable cloud connection (confirmed via
# Microsoft docs: cloud connection mapping is portal-only for Fabric
# Lakehouse data sources).
#
# After this one-time setup, all future report-only deployments preserve
# the connection automatically.
# ============================================================================

print("\nPHASE 6: Post-Deployment Auto-Configuration")
print("-" * 70)

if not is_first_deployment:
    print("  SKIPPED -- report-only deployment, connection already configured")
    print("  No manual steps required")
else:
    # --- Discover UAT dataset ID from pipeline artifacts --------------------
    print("  Step 1: Discovering UAT dataset ID...")
    uat_artifacts_url = f"{BASE_URL}/pipelines/{pipeline_id}/stages/1/artifacts"
    r = requests.get(uat_artifacts_url, headers=api_headers)
    r.raise_for_status()
    uat_artifacts = r.json()

    uat_datasets = uat_artifacts.get('datasets', [])
    if not uat_datasets:
        raise Exception("No datasets found in UAT stage -- deployment may have failed")

    uat_dataset_id = uat_datasets[0]['artifactId']
    uat_dataset_name = uat_datasets[0].get('artifactDisplayName', 'N/A')
    print(f"  UAT Dataset: {uat_dataset_name}")
    print(f"  UAT Dataset ID: {uat_dataset_id}")

    # --- Service principal takes over ---------------------------------------
    print("\n  Step 2: Service principal taking over...")
    takeover_url = f"{BASE_URL}/groups/{UAT_WORKSPACE_ID}/datasets/{uat_dataset_id}/Default.TakeOver"
    r = requests.post(takeover_url, headers=api_headers)

    if r.status_code == 200:
        print(f"  TakeOver: Succeeded")
    else:
        print(f"  TakeOver: {r.status_code} -- {r.text[:300]}")
        raise Exception("TakeOver failed -- check service principal permissions")

    # --- Bind to cloud connection (best effort) -----------------------------
    print("\n  Step 3: Binding to cloud connection...")
    bind_url = f"{BASE_URL}/groups/{UAT_WORKSPACE_ID}/datasets/{uat_dataset_id}/Default.BindToGateway"
    bind_body = {
        "gatewayObjectId": UAT_GATEWAY_ID,
        "datasourceObjectIds": [UAT_DATASOURCE_ID]
    }
    r = requests.post(bind_url, headers=api_headers, json=bind_body)

    if r.status_code == 200:
        print(f"  Bind: Succeeded")
    else:
        print(f"  Bind: {r.status_code} -- {r.text[:300]}")
        print(f"  Note: Bind may not map to named cloud connection. Complete manually.")

    # --- Manual step required -----------------------------------------------
    print(f"\n  MANUAL STEP REQUIRED (one-time only):")
    print(f"  1. Go to UAT workspace: AnalyticsBI_UAT")
    print(f"  2. Semantic model Settings: {uat_dataset_name}")
    print(f"  3. Gateway and cloud connections")
    print(f"  4. Maps to: select LH_Marketing_CRM")
    print(f"  5. Apply")
    print(f"\n  After this, all future deployments are fully automated.")

StatementMeta(, 3d77b0e9-9802-4788-a5d5-9aeb4fc047cb, 48, Finished, Available, Finished, False)


PHASE 6: Post-Deployment Auto-Configuration
----------------------------------------------------------------------
  SKIPPED -- report-only deployment, connection already configured
  No manual steps required


In [ ]:
###############################################################################
# CELL 8: PHASE 7 — POST-DEPLOYMENT VERIFICATION
###############################################################################

print("\nPHASE 7: Post-Deployment Verification")
print("-" * 70)

# --- Check UAT stage (Stage 1) ---------------------------------------------
artifacts_url = f"{BASE_URL}/pipelines/{pipeline_id}/stages/1/artifacts"
response = requests.get(artifacts_url, headers=api_headers)

if response.status_code == 200:
    data = response.json()

    for item_type in ['datasets', 'reports']:
        items = data.get(item_type, [])
        if items:
            print(f"\n  {item_type.upper()} now in UAT:")
            for item in items:
                print(f"    {item.get('artifactDisplayName', 'N/A')} | ID: {item.get('artifactId', 'N/A')}")
        else:
            print(f"\n  {item_type.upper()}: None found in UAT")
else:
    print(f"  Could not verify: {response.status_code}")

print(f"\n{'=' * 70}")
print("DEPLOYMENT COMPLETE")
print(f"{'=' * 70}")

StatementMeta(, 3d77b0e9-9802-4788-a5d5-9aeb4fc047cb, 49, Finished, Available, Finished, False)


PHASE 7: Post-Deployment Verification
----------------------------------------------------------------------

  DATASETS now in UAT:
    06.Lakehouse View to PBI Version 04 | ID: a448bba1-9a3f-4460-abbc-9e9d236e2ddf

  REPORTS now in UAT:
    06.Lakehouse View to PBI Version 04 | ID: c886de1e-4e2b-47aa-aee2-a47683d644b9

DEPLOYMENT COMPLETE


In [ ]:
###############################################################################
# CELL 9: PHASE 8 — DEPLOYMENT LOG
###############################################################################

print("\nPHASE 8: Deployment Log")
print("-" * 70)

# --- Get all pipeline operations (deployment history) -----------------------
operations_url = f"{BASE_URL}/pipelines/{pipeline_id}/operations"
response = requests.get(operations_url, headers=api_headers)

if response.status_code == 200:
    operations = response.json().get('value', [])

    if operations:
        log_data = []

        for op in operations:
            steps = op.get('executionPlan', {}).get('steps', [])

            if steps:
                for step in steps:
                    src_target = step.get('sourceAndTarget', {})
                    error      = step.get('error', {})

                    log_data.append({
                        'Operation ID':   op.get('id'),
                        'Pipeline Name':  PIPELINE_NAME,
                        'Status':         op.get('status'),
                        'Source Stage':   op.get('sourceStageOrder'),
                        'Target Stage':   op.get('targetStageOrder'),
                        'Started':        op.get('executionStartTime'),
                        'Ended':          op.get('executionEndTime'),
                        'Item Name':      src_target.get('sourceDisplayName', 'N/A'),
                        'Item Type':      step.get('type', 'N/A'),
                        'Step Status':    step.get('status', 'N/A'),
                        'Source Item ID': src_target.get('source', 'N/A'),
                        'Target Item ID': src_target.get('target', 'N/A'),
                        'Error Code':     error.get('errorCode', ''),
                        'Error Details':  error.get('errorDetails', ''),
                        'Deployment Note': op.get('note', {}).get('content', '') if op.get('note') else '',
                        'Log Extracted':  datetime.now().strftime('%d/%m/%Y %H:%M:%S')
                    })
            else:
                log_data.append({
                    'Operation ID':   op.get('id'),
                    'Pipeline Name':  PIPELINE_NAME,
                    'Status':         op.get('status'),
                    'Source Stage':   op.get('sourceStageOrder'),
                    'Target Stage':   op.get('targetStageOrder'),
                    'Started':        op.get('executionStartTime'),
                    'Ended':          op.get('executionEndTime'),
                    'Item Name':      'N/A',
                    'Item Type':      'N/A',
                    'Step Status':    'N/A',
                    'Source Item ID': 'N/A',
                    'Target Item ID': 'N/A',
                    'Error Code':     '',
                    'Error Details':  '',
                    'Deployment Note': op.get('note', {}).get('content', '') if op.get('note') else '',
                    'Log Extracted':  datetime.now().strftime('%d/%m/%Y %H:%M:%S')
                })

        df_log = pd.DataFrame(log_data)

        print(f"  {len(operations)} deployment(s) found")
        print(f"  {len(log_data)} total steps recorded")

        print(f"\n  DEPLOYMENT HISTORY:")
        print(f"  {'-' * 60}")
        for op in operations:
            print(f"  Operation : {op.get('id')}")
            print(f"  Status    : {op.get('status')}")
            print(f"  Started   : {op.get('executionStartTime')}")
            print(f"  Ended     : {op.get('executionEndTime')}")
            print(f"  Direction : Stage {op.get('sourceStageOrder')} to Stage {op.get('targetStageOrder')}")
            print(f"  {'-' * 60}")

        display(df_log)
    else:
        print("  No deployment history found")
else:
    print(f"  Could not retrieve history: {response.status_code}")
    print(f"  {response.text[:300]}")

print(f"\n{'=' * 70}")
print("LOGGING COMPLETE")
print(f"{'=' * 70}")

StatementMeta(, 3d77b0e9-9802-4788-a5d5-9aeb4fc047cb, 50, Finished, Available, Finished, False)


PHASE 8: Deployment Log
----------------------------------------------------------------------
  13 deployment(s) found
  13 total steps recorded

  DEPLOYMENT HISTORY:
  ------------------------------------------------------------
  Operation : e9d807ee-fae1-4b0c-88b9-b7fbddf5af64
  Status    : Succeeded
  Started   : 2026-02-22T15:54:34.773Z
  Ended     : 2026-02-22T15:54:35.9Z
  Direction : Stage 0 to Stage 1
  ------------------------------------------------------------
  Operation : bf099601-651e-4b1f-af63-0021c427af72
  Status    : Succeeded
  Started   : 2026-02-22T15:29:06.397Z
  Ended     : 2026-02-22T15:29:22.033Z
  Direction : Stage 0 to Stage 1
  ------------------------------------------------------------
  Operation : c242bb3e-d34b-4f02-8a21-b6151776fa7c
  Status    : Succeeded
  Started   : 2026-02-22T15:22:31.627Z
  Ended     : 2026-02-22T15:22:47.013Z
  Direction : Stage 0 to Stage 1
  ------------------------------------------------------------
  Operation : b5086dda

SynapseWidget(Synapse.DataFrame, 364225f8-2f22-4ce5-acd6-411136c3ace8)


LOGGING COMPLETE
